# HMMModel Implementation for Comparing Original FB Model Against Trained Baum-Welch Model

TThis notebook is the implementation for Module 10's extension of our HMM model: Baum-Welch. Baum-Welch is an Expectation-Maximization (EM) algorithm that takes an HMM with some initial parameters and iteratively improves them to better explain an observed sequence. The initial parameters can come from one of three approaches:

* Random initialization — probabilities are randomly generated and normalized, often used when no prior knowledge is available. Because EM can get stuck in local optima, this approach is typically run multiple times with different random seeds, keeping the best result.
* Uniform initialization — all probabilities are set equally, allowing the data to pull them toward where they need to be.
* Informed initialization — reasonable starting guesses based on domain knowledge, such as original model's probabilities

The flexibility in initialization exists because regardless of where the parameters start, Baum-Welch progressively improves them over successive iterations, eventually converging on the optimum of the likelihood function. Our implementation uses informed initialization, starting with the same initial, transition, and emission probabilities used to build our original model.

To evaluate the effect of training, we will run the Forward algorithm both before and after Baum-Welch to compare the original and trained model's log-likelihood scores. A higher score after training would confirm the model better explains the observed sequence than the original model does. We will also run Viterbi both before and after to compare the original and trained model's prediction of the overall optimal hidden state path. Similarly, Forward-Backward will be run before and after to compare the most likely state at each individual position across both models.

Rather than rewriting any of our previous work, we successfully extended our existing HMMModel class with a fully compatible Baum-Welch method. This was a deliberate design strategy to keep our implementation as clean and straightforward as possible. Built on the principles of Object-Oriented Programming (OOP), each week's addition has been implemented as a new method of the same HMMModel class, resulting in a highly modular, logically coherent program that extends with minimal revision and completes all expected tasks.





### Set Working Directory

In [4]:
import os

print("Current working directory:", os.getcwd())

Current working directory: /Users/biotechiestefnie/Desktop/HMM-Class-of-Algorithms/project10


## Imports and Notebook Formatting Helper Functions:
The following cell imports all packages and modules needed to call our script and supporting module for execution in this notebook. The first helper function included below the imports is used only for formatting the output matrices produced during execution, strictly for visualization purposes in Jupyter Notebooks and in the GitHub repository for this project, HMM_Class_of_Algorithms. The second helper function adds headers to the numpy array output matrices for interpretation of results.

In [ ]:
# Import packages and modules
import numpy as np
import copy
from project10 import HMMModel
from dict_maker import StrMatrix
from IPython.display import display, HTML
from tabulate import tabulate


def add_headers(matrix, states, observation):
    """
    Add state labels and observation characters as headers for a DP matrix. This is for
    forward and backward algorithms to display headers and hidden states because they are
    numpy arrays.
    Steps:
         Create a header row containing a blank cell followed by each observed character.
         Create one row per hidden state, placing the state label in column 0.
         Append the corresponding matrix row values after the state label.
         Return the combined header row and state-labeled rows as a nested list.
    """
    header_row = [" "] + list(observation)
    rows = []

    for i, state in enumerate(states):
        rows.append([state] + list(matrix[i]))

    return [header_row] + rows


def format_matrix(matrix, states, observation):
    """
    Prepare a DP matrix for HTML display by converting the NumPy array to a list-of-lists
    and adding state/observation headers. This function does not display the matrix; it
    returns the matrix with headers so that show_matrix_html can format and render it.
    Parameters:
         matrix (np.array): DP matrix from Viterbi, Forward, Backward, or FB algorithms
         states (list or np.array): hidden state labels in model order
         observation (str): observed sequence used to label columns
    Returns:
         list: matrix with headers suitable for show_matrix_html
    """
    matrix_list = matrix.tolist()

    return add_headers(matrix_list, states, observation)


def show_matrix_html(matrix, float_format="{:.3e}"):
    """
    Function to display matrix output from execution of project09.py in an organized, interpretable format within a Jupyter notebook.
    Parameters:
         matrix (np.array): output matrix from each HMMModel method execution
         float_format (str, optional): formatting string for floating point output
    """
    formatted = []

    for row in matrix:
        new_row = []

        for cell in row:

            if isinstance(cell, float):
                new_row.append(float_format.format(cell))

            else:
                new_row.append(str(cell))

        formatted.append(new_row)

    return HTML(tabulate(formatted, headers="firstrow", tablefmt="html"))



### Define Initial Parameters

In [6]:
# Example observation sequences
obs1 = "GGCACTGAA"
obs2 = "ATGCAATGC"
obs3 = "AATGCCTGA"
obs = [obs1, obs2, obs3]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "H": 0.5,  # H = High GC content state
    "L": 0.5   # L = Low GC content state
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "H": {"H": 0.6, "L": 0.4},
    "L": {"H": 0.3, "L": 0.7}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "H": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
    "L": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

### Example StrMatrix Construction from Initial Parameters Emission Dictionary

In [7]:

x = StrMatrix(emit_probs)

# Format matrix with headers
headered = format_matrix(
    x.get_matrix(),
    states=x.row_labels,
    observation=x.col_labels
)

# Display with tabulate using first row as header
display(show_matrix_html(headered))


,A,C,G,T
H,-1.609,-1.204,-1.204,-1.609
L,-1.204,-1.609,-1.609,-1.204


### Demonstration of StrMatrix with Viterbi

In [8]:
# StrMatrix demonstration for Viterbi (I/G)
trans_v = StrMatrix(trans_probs)
emit_v = StrMatrix(emit_probs)

# Format matrices with headers
trans_headered = format_matrix(
    trans_v.get_matrix(),
    states=trans_v.row_labels,
    observation=trans_v.col_labels
)

emit_headered = format_matrix(
    emit_v.get_matrix(),
    states=emit_v.row_labels,
    observation=emit_v.col_labels
)

# Display with headers
display(show_matrix_html(trans_headered))
display(show_matrix_html(emit_headered))


,H,L
H,-0.5108,-0.9163
L,-1.204,-0.3567


,A,C,G,T
H,-1.609,-1.204,-1.204,-1.609
L,-1.204,-1.609,-1.609,-1.204


### Demonstration of StrMatrix with Forward/Backward

In [9]:
# StrMatrix demonstration for Forward–Backward
trans_fb = StrMatrix(trans_probs)
emit_fb = StrMatrix(emit_probs)

# Format matrices with headers
trans_fb_headered = format_matrix(
    trans_fb.get_matrix(),
    states=trans_fb.row_labels,
    observation=trans_fb.col_labels
)

emit_fb_headered = format_matrix(
    emit_fb.get_matrix(),
    states=emit_fb.row_labels,
    observation=emit_fb.col_labels
)

# Display with headers
display(show_matrix_html(trans_fb_headered))
display(show_matrix_html(emit_fb_headered))


,H,L
H,-0.5108,-0.9163
L,-1.204,-0.3567


,A,C,G,T
H,-1.609,-1.204,-1.204,-1.609
L,-1.204,-1.609,-1.609,-1.204


## Instantiate Models
This cell creates the original model object then used Python's Deep Copy to create an identical copy of original_matrix, renamed to trained_matrix. This ensures we can keep original_model with its own parameters to compare and still run our Baum-Welch, which writes over probabilities for each iteration.

In [10]:
# Original Model
original_model = HMMModel(init_probs, trans_probs, emit_probs)

# Deep copy model to create trained model instance before training
trained_model = copy.deepcopy(original_model)


## Execute HMMModel for Original Model

In [11]:
# Run all three algorithms on original model
vmat1, tmat1, path1 = original_model.viterbi_algorithm(obs1)
fwd_matrix1, log_likelihood1 = original_model.forward_algorithm(obs1)
most_likely_states1, fb_matrix1 = original_model.forward_backward_algorithm(obs1)

vmat2, tmat2, path2 = original_model.viterbi_algorithm(obs2)
fwd_matrix2, log_likelihood2 = original_model.forward_algorithm(obs2)
most_likely_states2, fb_matrix2 = original_model.forward_backward_algorithm(obs2)

vmat3, tmat3, path3 = original_model.viterbi_algorithm(obs3)
fwd_matrix3, log_likelihood3 = original_model.forward_algorithm(obs3)
most_likely_states3, fb_matrix3 = original_model.forward_backward_algorithm(obs3)

### Execute Trained Model (Baum-Welch) on Each Observation Sequence

In [12]:
# Execute Baum-Welch to train model on each observation sequentially
new_init1, new_trans1, new_emit1 = trained_model.baumwelch_algorithm(obs1, n_iter=25)
new_init2, new_trans2, new_emit2 = trained_model.baumwelch_algorithm(obs2, n_iter=25)
new_init3, new_trans3, new_emit3 = trained_model.baumwelch_algorithm(obs3, n_iter=25)

display(new_init1, new_trans1, new_emit1)
display(new_init2, new_trans2, new_emit2)
display(new_init3, new_trans3, new_emit3)

{np.str_('H'): 1.0, np.str_('L'): 6.342465746313852e-74}

{np.str_('H'): {np.str_('H'): 0.333339993802565,
  np.str_('L'): 0.6666554292600568},
 np.str_('L'): {np.str_('H'): 0.2000041858991319,
  np.str_('L'): 0.7999985603309827}}

{np.str_('H'): {'A': 1.0020168734467061e-06,
  'C': 1.926527305816046e-05,
  'G': 0.9999797326169733,
  'T': 9.309490481960477e-11},
 np.str_('L'): {'A': 0.5000034217508789,
  'C': 0.3333263156508231,
  'G': 2.2883878107765658e-06,
  'T': 0.16666797421048732}}

{np.str_('H'): 1.0, np.str_('L'): 2.544569195233944e-63}

{np.str_('H'): {np.str_('H'): 1.3037404091880257e-23,
  np.str_('L'): 1.2163891482285472},
 np.str_('L'): {np.str_('H'): 0.5471773732700068,
  np.str_('L'): 0.28659189614667996}}

{np.str_('H'): {'A': 0.5655138722788877,
  'C': 1.3769583786751045e-20,
  'G': 0.27970741505414554,
  'T': 0.15477871266696672},
 np.str_('L'): {'A': 0.18725797022605875,
  'C': 0.362032354238621,
  'G': 0.1860556664275353,
  'T': 0.264654009107785}}

{np.str_('H'): 1.0, np.str_('L'): 1.2362072497697903e-76}

{np.str_('H'): {np.str_('H'): 1.2425700185004502e-12,
  np.str_('L'): 0.9999999999987567},
 np.str_('L'): {np.str_('H'): 9.05664932723065e-06,
  np.str_('L'): 0.999990943350673}}

{np.str_('H'): {'A': 1.0,
  'C': 2.2137350808468673e-51,
  'G': 1.1786833013145375e-36,
  'T': 4.657725506550846e-35},
 np.str_('L'): {'A': 0.24999405652666307,
  'C': 0.250001981157779,
  'G': 0.250001981157779,
  'T': 0.250001981157779}}

### Run Viterbi, Forward, and Forward-Backward with Trained Results to Obtain Predictions for Comparison

In [13]:
# Run all three algorithms on trained model
vmat1_t, tmat1_t, path1_t = trained_model.viterbi_algorithm(obs1)
fwd_matrix1_t, log_likelihood1_t = trained_model.forward_algorithm(obs1)
most_likely_states1_t, fb_matrix1_t = trained_model.forward_backward_algorithm(obs1)

vmat2_t, tmat2_t, path2_t = trained_model.viterbi_algorithm(obs2)
fwd_matrix2_t, log_likelihood2_t = trained_model.forward_algorithm(obs2)
most_likely_states2_t, fb_matrix2_t = trained_model.forward_backward_algorithm(obs2)

vmat3_t, tmat3_t, path3_t = trained_model.viterbi_algorithm(obs3)
fwd_matrix3_t, log_likelihood3_t = trained_model.forward_algorithm(obs3)
most_likely_states3_t, fb_matrix3_t = trained_model.forward_backward_algorithm(obs3)

### Display Results of Each Observation for Original Model

In [14]:
# Display original model results for obs1
print("========> * ORIGINAL MODEL - OBS1 * <========")
print("Viterbi Path:", path1)
display(show_matrix_html(format_matrix(vmat1, original_model.states, obs1)))
display(HTML("<br>"))
print("Log Likelihood:", log_likelihood1)
print("Most Likely States:", most_likely_states1)
display(show_matrix_html(format_matrix(fb_matrix1, original_model.states, obs1)))
display(HTML("<br>"))

# Display original model results for obs2
print("========> * ORIGINAL MODEL - OBS2 * <========")
print("Viterbi Path:", path2)
display(show_matrix_html(format_matrix(vmat2, original_model.states, obs2)))
display(HTML("<br>"))
print("Log Likelihood:", log_likelihood2)
print("Most Likely States:", most_likely_states2)
display(show_matrix_html(format_matrix(fb_matrix2, original_model.states, obs2)))
display(HTML("<br>"))

# Display original model results for obs3
print("========> * ORIGINAL MODEL - OBS3 * <========")
print("Viterbi Path:", path3)
display(show_matrix_html(format_matrix(vmat3, original_model.states, obs3)))
display(HTML("<br>"))
print("Log Likelihood:", log_likelihood3)
print("Most Likely States:", most_likely_states3)
display(show_matrix_html(format_matrix(fb_matrix3, original_model.states, obs3)))

========> * ORIGINAL MODEL - OBS1 * <========
Viterbi Path: ['H', 'H', 'H', 'L', 'L', 'L', 'L', 'L', 'L']


,G,G,C,A,C,T,G,A,A
H,-1.897,-3.612,-5.327,-7.447,-9.162,-11.28,-13,-15.12,-17.24
L,-2.303,-4.269,-6.138,-7.447,-9.413,-10.97,-12.94,-14.5,-16.06


Log Likelihood: -12.483820056859475
Most Likely States: ['H' 'H' 'H' 'L' 'H' 'L' 'H' 'L' 'L']


,G,G,C,A,C,T,G,A,A
H,-0.4947,-0.5306,-0.5618,-1.049,-0.6827,-1.097,-0.6984,-1.113,-1.16
L,-1.068,-1.007,-0.959,-0.4426,-0.8275,-0.4124,-0.8142,-0.3982,-0.3761


========> * ORIGINAL MODEL - OBS2 * <========
Viterbi Path: ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']


...

,A,T,G,C,A,A,T,G,C
H,-0.9434,-1.116,-0.6229,-0.5593,-1.041,-1.184,-1.188,-0.6516,-0.6011
L,-0.3991,-0.3111,-0.7584,-0.8365,-0.3391,-0.2748,-0.2733,-0.7365,-0.7945


========> * ORIGINAL MODEL - OBS3 * <========
Viterbi Path: ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']


,A,A,T,G,C,C,T,G,A
H,-2.303,-4.423,-6.271,-7.426,-9.141,-10.86,-12.98,-14.69,-16.81
L,-1.897,-3.458,-5.018,-6.985,-8.951,-10.92,-12.48,-14.44,-16


Log Likelihood: -12.490497134887951
Most Likely States: ['L' 'L' 'L' 'H' 'H' 'H' 'L' 'H' 'L']


,A,A,T,G,C,C,T,G,A
H,-0.9891,-1.194,-1.212,-0.6598,-0.5661,-0.56,-1.034,-0.6491,-1.032
L,-0.4286,-0.3282,-0.3208,-0.793,-0.9121,-0.9208,-0.4241,-0.8188,-0.4404


### Display Results for Trained Model

In [15]:
# Display trained model results for obs1
print("========> * TRAINED MODEL - OBS1 * <========")
print("Viterbi Path:", path1_t)
display(show_matrix_html(format_matrix(vmat1_t, trained_model.states, obs1)))
print("Log Likelihood:", log_likelihood1_t)
print("Most Likely States:", most_likely_states1_t)
display(show_matrix_html(format_matrix(fb_matrix1_t, trained_model.states, obs1)))
display(HTML("<br>"))

# Display trained model results for obs2
print("========> * TRAINED MODEL - OBS2 * <========")
print("Viterbi Path:", path2_t)
display(show_matrix_html(format_matrix(vmat2_t, trained_model.states, obs2)))
print("Log Likelihood:", log_likelihood2_t)
print("Most Likely States:", most_likely_states2_t)
display(show_matrix_html(format_matrix(fb_matrix2_t, trained_model.states, obs2)))
display(HTML("<br>"))

# Display trained model results for obs3
print("========> * TRAINED MODEL - OBS3 * <========")
print("Viterbi Path:", path3_t)
display(show_matrix_html(format_matrix(vmat3_t, trained_model.states, obs3)))
print("Log Likelihood:", log_likelihood3_t)
print("Most Likely States:", most_likely_states3_t)
display(show_matrix_html(format_matrix(fb_matrix3_t, trained_model.states, obs3)))


========> * TRAINED MODEL - OBS1 * <========
Viterbi Path: ['H', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']


,G,G,C,A,C,T,G,A,A
H,-82.73,-192.9,-212.4,-97.11,-215.1,-178.9,-184,-102.7,-104
L,-176.2,-84.11,-85.5,-86.89,-88.27,-89.66,-91.05,-92.43,-93.82


Log Likelihood: -93.81900668325856
Most Likely States: ['H' 'L' 'L' 'L' 'L' 'L' 'L' 'L' 'L']


,G,G,C,A,C,T,G,A,A
H,-4.527e-06,-108.8,-126.9,-10.23,-126.9,-89.28,-92.95,-10.23,-10.23
L,-93.44,-4.527e-06,-4.527e-06,-3.623e-05,-4.527e-06,-4.527e-06,-4.527e-06,-3.623e-05,-3.623e-05


========> * TRAINED MODEL - OBS2 * <========
Viterbi Path: ['H', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']


...

,A,T,G,C,A,A,T,G,C
H,-3.17e-05,-105.1,-92.95,-126.9,-10.23,-10.23,-89.28,-92.95,-126.9
L,-176.2,0,0,0,-3.17e-05,-3.17e-05,0,0,0


========> * TRAINED MODEL - OBS3 * <========
Viterbi Path: ['H', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']


,A,A,T,G,C,C,T,G,A
H,0,-27.41,-92.05,-97.11,-132.4,-133.8,-97.6,-102.7,-21.32
L,-176.2,-1.386,-2.773,-4.159,-5.545,-6.932,-8.318,-9.704,-11.09


Log Likelihood: -11.090382059410905
Most Likely States: ['H' 'L' 'L' 'L' 'L' 'L' 'L' 'L' 'L']


,A,A,T,G,C,C,T,G,A
H,3.281e-10,-26.03,-89.28,-92.95,-126.9,-126.9,-89.28,-92.95,-10.23
L,-176.2,3.231e-10,-4.528e-06,-4.528e-06,-4.528e-06,-4.528e-06,-4.528e-06,-4.528e-06,-3.623e-05
